# Customer Support Agent — Evaluation, LangSmith Observability & OpenTelemetry

**Goal:** Build a small e-commerce customer support agent with **LangGraph**, trace every run in **LangSmith**, and learn how to evaluate it like a real ML system.

By the end of this notebook you will have:
1. A LangGraph agent that classifies support tickets into clear categories
2. A synthetic ticket dataset with ground-truth labels
3. LangSmith traces for every prediction (clickable in the UI), emitted through OpenTelemetry
4. Accuracy / precision / recall metrics
5. A CSV of results you can open in a spreadsheet and annotate
6. A simple loop: **inspect failures → tweak the prompt → re-run → compare**

---

## The workflow you'll follow

1. **Run the agent** on the synthetic tickets.
2. **Look at the metrics** and open the exported CSV in Google Sheets / Excel.
3. **Add validator comments** in the `validator_comment` column — flag anything that looks wrong, ambiguous, or surprising.
4. **Cluster the failures** into 2–4 *failure categories* (e.g., "misroutes complaints as questions", "confuses refund vs. return").
5. **Pick the one failure category that matters most** for your use case.
6. **Tweak the classifier prompt** at the bottom of the notebook to address it.
7. **Re-run** the agent and compare the new metrics + LangSmith/OpenTelemetry traces.

## 1. Install dependencies

In [ ]:
%pip install --quiet langgraph "langsmith[otel]>=0.4.25" langchain-openai langchain-core pandas scikit-learn pydantic tqdm python-dotenv opentelemetry-sdk opentelemetry-exporter-otlp


## 2. Set up API keys, LangSmith tracing, and OpenTelemetry

You need two keys:
- **`OPENAI_API_KEY`** — get one at [platform.openai.com/api-keys](https://platform.openai.com/api-keys)
- **`LANGSMITH_API_KEY`** — get one at [smith.langchain.com](https://smith.langchain.com) → Settings → API Keys

Once these are set, every LLM call is traced in LangSmith. We also enable OpenTelemetry so each ticket-level eval run can carry standard span metadata like ticket id, prompt version, expected label, predicted label, and correctness.

In [ ]:
import os
import getpass
from dotenv import load_dotenv

# Clear stale tracing variables so an old endpoint does not make OTel export to a 404 URL.
for k in [
    "LANGCHAIN_API_KEY",
    "LANGCHAIN_ENDPOINT",
    "LANGCHAIN_TRACING_V2",
    "OTEL_EXPORTER_OTLP_ENDPOINT",
    "OTEL_EXPORTER_OTLP_TRACES_ENDPOINT",
    "OTEL_EXPORTER_OTLP_HEADERS",
    "OTEL_EXPORTER_OTLP_TRACES_HEADERS",
    "OTEL_EXPORTER_OTLP_PROTOCOL",
]:
    os.environ.pop(k, None)

# Reload the current project-local keys after clearing stale process values.
load_dotenv(override=True)

def _set(key: str, prompt: str):
    if not os.environ.get(key):
        os.environ[key] = getpass.getpass(prompt)

_set("OPENAI_API_KEY",    "OpenAI API key: ")
_set("LANGSMITH_API_KEY", "LangSmith API key: ")

# LangSmith is still the evaluation/debugging UI.
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"] = os.getenv(
    "LANGSMITH_PROJECT", "CustomerAgentEvaluationAndLLM-As-Judge"
)
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"  # use https://eu.api.smith.langchain.com for EU accounts

# OpenTelemetry is the standard tracing layer. LangSmith receives the emitted spans.
os.environ["LANGSMITH_OTEL_ENABLED"] = "true"
os.environ["OTEL_SERVICE_NAME"] = "CustomerAgentEvaluationAndLLM-As-Judge"
otel_endpoint = "https://api.smith.langchain.com/otel/v1/traces"

os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = otel_endpoint
os.environ["OTEL_EXPORTER_OTLP_TRACES_ENDPOINT"] = otel_endpoint
os.environ["OTEL_EXPORTER_OTLP_PROTOCOL"] = "http/protobuf"

otel_headers = f"x-api-key={os.environ['LANGSMITH_API_KEY']},Langsmith-Project={os.environ['LANGSMITH_PROJECT']}"
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = otel_headers
os.environ["OTEL_EXPORTER_OTLP_TRACES_HEADERS"] = otel_headers

print("Traces will appear in LangSmith project:", os.environ["LANGSMITH_PROJECT"])
print("OpenTelemetry service name:", os.environ["OTEL_SERVICE_NAME"])
print("OpenTelemetry traces endpoint:", os.environ["OTEL_EXPORTER_OTLP_TRACES_ENDPOINT"])


## 3. Define the ticket categories

We keep the label space small and unambiguous on purpose — when there are 50 categories, *everything* looks like a model failure. Five is a good teaching size.

| Category | What belongs here |
|---|---|
| `order_status` | "Where is my order?", tracking, delivery ETA |
| `refund_request` | Customer wants money back, return-for-refund |
| `product_issue` | Item arrived broken, wrong, defective, or not as described |
| `account_help` | Login, password, address, payment method changes |
| `other` | Anything that doesn't fit above (general questions, feedback) |

In [ ]:
CATEGORIES = [
    "order_status",
    "refund_request",
    "product_issue",
    "account_help",
    "other",
]

## 4. Synthetically generate the ticket dataset

We hand-write a small seed set of tickets with **ground-truth labels**, then ask the LLM to generate variations in the same style. This gives us a dataset that:
- has trustworthy labels (we wrote the seeds),
- contains realistic phrasing (the LLM paraphrases),
- includes some intentionally tricky/ambiguous cases (so the agent has something to fail on).

If you want a bigger or smaller dataset, change `EXPANSIONS_PER_SEED` below.

In [ ]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List

# Seed tickets — each one is a (message, true_category) pair we trust.
# Many of these are intentionally tricky: they mention keywords from other categories,
# use sarcasm, bury the real intent inside emotional language, or sit on a real edge case.
# The goal is to give the model something to actually fail on.
SEED_TICKETS = [
    # ---------- order_status ----------
    ("Hi, I ordered a blender 5 days ago and the tracking page hasn't updated. Can you tell me where it is?", "order_status"),
    ("I never received my order and I want my money back.", "order_status"),  # mentions refund but the real issue is non-delivery
    ("Order shows delivered last Tuesday but it's not at my door, my neighbor's, or the mailroom. What now?", "order_status"),
    ("The website said 2-day shipping. It's been 9 days. Are you kidding me?", "order_status"),  # sarcastic
    ("Tracking link in the email just spins forever. Order #88421.", "order_status"),  # easy to mis-call account_help
    # ---------- refund_request ----------
    ("I'd like to return the headphones I bought last week and get my money back. They're unopened.", "refund_request"),
    ("Where is my refund? I returned the item two weeks ago and still nothing on my card.", "refund_request"),
    ("Please cancel order 99021 and refund my card. I no longer need it.", "refund_request"),  # could feel like account_help
    ("I was charged $89 but the website showed $79 at checkout. Please refund the difference.", "refund_request"),
    ("Returning these for store credit is fine but honestly I'd prefer cash back to my original card.", "refund_request"),
    # ---------- product_issue ----------
    ("The coffee maker arrived with a cracked carafe. Really disappointed.", "product_issue"),
    ("My laptop arrived damaged and I want a full refund, not a replacement.", "product_issue"),  # asks for refund, but root cause is damage
    ("You sent me a size medium shirt but I ordered a large. Second time this has happened.", "product_issue"),
    ("The product description said 'wireless' but I had to buy a separate dongle to use it. Misleading.", "product_issue"),  # subtle "not as described"
    ("Item itself works fine but the box was crushed and the instruction manual is missing.", "product_issue"),
    # ---------- account_help ----------
    ("I can't log into my account — it keeps saying my password is wrong even after I reset it.", "account_help"),
    ("How do I update the credit card on file? I don't see the option anywhere in settings.", "account_help"),
    ("The website won't let me check out — it keeps logging me out mid-payment.", "account_help"),  # sounds like a site bug, technically account
    ("Please remove my old shipping address. I moved last month and don't want stuff going there.", "account_help"),
    ("I keep getting 2FA codes I didn't request. Is someone trying to access my account?", "account_help"),
    # ---------- other ----------
    ("Do you guys ship to Canada? Couldn't find it on the FAQ page.", "other"),
    ("Just wanted to say the customer service rep I spoke to yesterday was amazing. Thank you!", "other"),
    ("Is the red version of SKU-1140 back in stock?", "other"),
    ("Do you offer a student discount? Couldn't find one at checkout.", "other"),
    ("When's your next sale? My birthday is coming up and I'd love to splurge.", "other"),
]

EXPANSIONS_PER_SEED = 3  # 25 seeds * (1 + 3) = 100 tickets total

generator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.9)

class Paraphrases(BaseModel):
    variations: List[str] = Field(description="Realistic paraphrases of the original ticket")

def expand(seed_text: str, n: int) -> List[str]:
    if n <= 0:
        return []
    structured = generator_llm.with_structured_output(Paraphrases)
    out = structured.invoke(
        f"You write realistic e-commerce customer-support tickets.\n"
        f"Produce {n} short paraphrases of the ticket below. KEEP THE SAME UNDERLYING INTENT, "
        f"but vary tone (frustrated, polite, terse, rambling, sarcastic), names, order numbers, "
        f"products, and the specific phrasing. If the original is ambiguous or mentions keywords "
        f"from multiple categories, preserve that ambiguity — don't sanitize it.\n\n"
        f"Original ticket:\n{seed_text}"
    )
    return out.variations

tickets = []
for text, label in SEED_TICKETS:
    tickets.append({"id": f"t{len(tickets):03d}", "text": text, "true_category": label, "source": "seed"})
    for variation in expand(text, EXPANSIONS_PER_SEED):
        tickets.append({"id": f"t{len(tickets):03d}", "text": variation, "true_category": label, "source": "synthetic"})

print(f"Generated {len(tickets)} tickets across {len(CATEGORIES)} categories.")
from collections import Counter
for cat, n in Counter(t["true_category"] for t in tickets).items():
    print(f"  {cat:>15}: {n}")
print()
for t in tickets[:5]:
    print(f"  [{t['true_category']:>15}] {t['text'][:90]}")

## 5a. What OpenTelemetry adds here

LangSmith is the place where we inspect AI traces and compare baseline vs improved runs. OpenTelemetry is the standard way we emit structured traces from code.

In this notebook, we add one OpenTelemetry parent span around each ticket classification. The LangGraph / LangChain internals still create child spans for the model call, and the parent span carries eval metadata:

- `eval.example_id`: ticket id
- `eval.run_name`: baseline or improved
- `eval.prompt_version`: v1 or v2
- `eval.true_category`: ground-truth label
- `eval.predicted_category`: model output
- `eval.correct`: whether the prediction matched the label
- `eval.reasoning`: model explanation

This makes each row in the CSV traceable back to a specific LangSmith/OpenTelemetry run.

If you see `Failed to export span batch code: 404`, the classifier is still running, but the OTLP exporter is pointed at the wrong URL. This notebook clears stale `OTEL_EXPORTER_*` values and explicitly sends trace spans to `https://api.smith.langchain.com/otel/v1/traces`.


## 5. Build the LangGraph classification agent

LangGraph models an agent as a **graph of nodes**. For a classifier, the graph is tiny — one node that calls the LLM with a structured output schema. We're using LangGraph here (instead of just calling the LLM directly) so the pattern scales to multi-step agents later (e.g., add a retrieval node, a tool-calling node, a confidence-check node).

Because LangSmith tracing is on, **every graph invocation becomes a clickable trace** showing each node's input/output.

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate

# --- THIS PROMPT IS WHAT YOU'LL TWEAK LATER ---
CLASSIFIER_PROMPT = """You are a triage system for an e-commerce support inbox.

Classify the customer's ticket into EXACTLY ONE of these categories:

- order_status: questions about where an order is, tracking, delivery ETA
- refund_request: the customer wants their money back
- product_issue: the item arrived broken, wrong, defective, or not as described
- account_help: login, password, address, payment method changes
- other: anything that doesn't fit the above (general questions, feedback, browsing)

Return only the category key.

Ticket:
{ticket_text}
"""

class Classification(BaseModel):
    category: Literal["order_status", "refund_request", "product_issue", "account_help", "other"]
    reasoning: str = Field(description="One short sentence explaining the choice.")

class AgentState(TypedDict):
    ticket_text: str
    category: str
    reasoning: str

def build_agent(prompt_template: str):
    """Compile a LangGraph agent. Re-call this any time you change the prompt."""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(Classification)
    prompt = ChatPromptTemplate.from_template(prompt_template)

    def classify_node(state: AgentState) -> AgentState:
        result = (prompt | llm).invoke({"ticket_text": state["ticket_text"]})
        return {"ticket_text": state["ticket_text"], "category": result.category, "reasoning": result.reasoning}

    graph = StateGraph(AgentState)
    graph.add_node("classify", classify_node)
    graph.add_edge(START, "classify")
    graph.add_edge("classify", END)
    return graph.compile()

agent = build_agent(CLASSIFIER_PROMPT)

# Smoke test on one ticket.
sample = agent.invoke({"ticket_text": tickets[0]["text"], "category": "", "reasoning": ""})
print("Ticket:    ", tickets[0]["text"])
print("Predicted: ", sample["category"])
print("Reasoning: ", sample["reasoning"])

In [ ]:
from opentelemetry import trace

# This tracer creates ticket-level spans. LangSmith receives them because
# LANGSMITH_OTEL_ENABLED=true is set above.
tracer = trace.get_tracer("CustomerAgentEvaluationAndLLM-As-Judge")


## 6. Run the agent on every ticket

Each invocation is automatically traced in LangSmith. After this cell finishes, go to your [LangSmith dashboard](https://smith.langchain.com) → project **`CustomerAgentEvaluationAndLLM-As-Judge`** → and you'll see every prediction with full input/output/latency.

In [ ]:
import pandas as pd
from tqdm import tqdm
from langsmith.integrations.otel import set_langsmith_metadata_attribute

def run_predictions(agent, tickets, run_name="baseline", prompt_version="v1") -> pd.DataFrame:
    rows = []
    for t in tqdm(tickets, desc=f"Classifying {run_name}"):
        # One parent span per ticket makes the spreadsheet row traceable in LangSmith.
        with tracer.start_as_current_span("customer_support.classify_ticket") as span:
            span.set_attribute("langsmith.span.kind", "chain")
            span.set_attribute("eval.run_name", run_name)
            span.set_attribute("eval.prompt_version", prompt_version)
            span.set_attribute("eval.example_id", t["id"])
            span.set_attribute("eval.true_category", t["true_category"])
            span.set_attribute("input.ticket_text", t["text"])

            out = agent.invoke({"ticket_text": t["text"], "category": "", "reasoning": ""})
            correct = t["true_category"] == out["category"]

            metadata = {
                "run_name": run_name,
                "prompt_version": prompt_version,
                "example_id": t["id"],
                "true_category": t["true_category"],
                "predicted_category": out["category"],
                "correct": correct,
                "reasoning": out["reasoning"],
            }

            for key, value in metadata.items():
                set_langsmith_metadata_attribute(span, key, value)

            span.set_attribute("eval.predicted_category", out["category"])
            span.set_attribute("eval.correct", correct)
            span.set_attribute("eval.reasoning", out["reasoning"])
            span.set_attribute("output.category", out["category"])

            rows.append({
                "id": t["id"],
                "ticket_text": t["text"],
                "true_category": t["true_category"],
                "predicted_category": out["category"],
                "reasoning": out["reasoning"],
                "correct": correct,
                "otel_run_name": run_name,
                "otel_prompt_version": prompt_version,
            })
    return pd.DataFrame(rows)

results_v1 = run_predictions(agent, tickets, run_name="baseline", prompt_version="v1")
results_v1.head(10)


## 7. Evaluate: accuracy, precision, recall

- **Accuracy** = fraction of tickets classified correctly overall.
- **Precision (per class)** = of all the tickets the model *called* `refund_request`, how many actually were? High precision = few false alarms.
- **Recall (per class)** = of all the tickets that *truly were* `refund_request`, how many did the model catch? High recall = few misses.

**Why look at both:** a model that *always* predicts `other` will have 20% accuracy and 100% recall on `other` but 0% recall on everything else. Per-class precision/recall exposes that immediately.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def evaluate(df: pd.DataFrame, label: str):
    y_true = df["true_category"]
    y_pred = df["predicted_category"]
    print(f"=== {label} ===")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.2%}  ({df['correct'].sum()}/{len(df)} correct)\n")
    print("Per-class precision / recall / F1:")
    print(classification_report(y_true, y_pred, labels=CATEGORIES, zero_division=0))
    print("Confusion matrix (rows = true, cols = predicted):")
    cm = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=CATEGORIES),
        index=CATEGORIES, columns=CATEGORIES,
    )
    print(cm)
    return accuracy_score(y_true, y_pred)

acc_v1 = evaluate(results_v1, "Run 1 — baseline prompt")

## 8. Export to spreadsheet for validator comments

This is the **human-in-the-loop** step. Open `results_v1.csv` in Google Sheets or Excel.

Each row has an empty `validator_comment` column. Your job:

1. **Filter `correct == FALSE`** to see only the failures.
2. For each failure, write a short note in `validator_comment` — e.g. *"complaint about delivery, model called it product_issue"*, *"ambiguous, I'd accept either"*, *"label is wrong, this really is `other`"*.
3. Also scan a sample of `correct == TRUE` rows — sometimes the model gets the right label for the *wrong reason*.
4. Once you've annotated, **cluster the comments into 2–4 failure categories** in a separate tab. For example:
   - *"Confuses `order_status` with `refund_request` when the customer mentions both delivery and money"*
   - *"Calls polite thank-you messages `account_help`"*
   - *"Routes 'wrong item' as `refund_request` instead of `product_issue`"*
5. **Pick the one failure category that matters most for your use case** (the one that's most expensive if it happens in production), and bring it back to step 9.

In [ ]:
results_v1_export = results_v1.copy()
results_v1_export["validator_comment"] = ""
results_v1_export["failure_category"] = ""
results_v1_export.to_csv("results_v1.csv", index=False)
print("Wrote results_v1.csv — open it in Google Sheets or Excel.")
print("Columns:", list(results_v1_export.columns))

## 9. Tweak the prompt to fix the failure category you picked

Edit `IMPROVED_PROMPT` below to address the failure you chose. Some common moves:

- **Add disambiguation rules.** *"If the customer mentions both delivery delay AND a refund request, classify as `order_status` — the refund is downstream of the delivery problem."*
- **Add a few-shot example** of the exact failure case with the correct label.
- **Tighten a category definition.** *"`account_help` is ONLY for login/password/profile issues, not for general site bugs."*
- **Force a step.** *"First identify the customer's primary intent in one sentence, then pick the category."*

Keep the change focused on the **one failure category** you picked. If you change everything, you won't know what helped.

### Selected failure category

Product issues misrouted as refund requests.

This failure matters most because damaged, defective, wrong, or misleading
products require quality investigation and appropriate product-support handling,
not only refund processing.

In [ ]:
IMPROVED_PROMPT = CLASSIFIER_PROMPT.replace(
    "Return only the category key.",
    """Disambiguation rule:
If an item is damaged, wrong, defective, missing an advertised capability,
or not as described, classify it as product_issue even when the customer
also requests a refund.

Return only the category key."""
)


agent_v2 = build_agent(IMPROVED_PROMPT)
results_v2 = run_predictions(agent_v2, tickets, run_name="improved", prompt_version="v2")

results_v2_export = results_v2.copy()
results_v2_export["validator_comment"] = ""
results_v2_export["failure_category"] = ""
results_v2_export.to_csv("results_v2.csv", index=False)

acc_v2 = evaluate(results_v2, "Run 2 — improved prompt")

## 10. Compare the two runs

Now look at the headline accuracy and at *which specific tickets flipped* between runs. Some will go from wrong → right (the win you were aiming for). Some may go from right → wrong (a regression you caused). This is normal — almost no prompt change is strictly Pareto-better, and the trade-offs are the most important thing to understand.

In [ ]:
print(f"Accuracy v1: {acc_v1:.2%}")
print(f"Accuracy v2: {acc_v2:.2%}")
print(f"Δ          : {(acc_v2 - acc_v1):+.2%}\n")

comparison = results_v1.merge(
    results_v2[["id", "predicted_category", "reasoning", "correct"]],
    on="id", suffixes=("_v1", "_v2"),
)

flipped = comparison[comparison["predicted_category_v1"] != comparison["predicted_category_v2"]]
print(f"{len(flipped)} tickets changed prediction between runs.\n")

wins   = flipped[(~flipped["correct_v1"]) & (flipped["correct_v2"])]
losses = flipped[(flipped["correct_v1"]) & (~flipped["correct_v2"])]
print(f"Wins (wrong → right):     {len(wins)}")
print(f"Regressions (right → wrong): {len(losses)}")

flipped[["ticket_text", "true_category", "predicted_category_v1", "predicted_category_v2"]]

## 11. What to take away

- **LangGraph** gave us a clean shape for the agent. Right now it's one node, but you can drop in retrieval, tools, or a self-check node without rewriting the eval harness.
- **LangSmith** turned every LLM call into an inspectable trace. OpenTelemetry added a standard parent span around each ticket so CSV rows, prompt versions, labels, and correctness are attached to the trace.
- **Accuracy alone is a trap.** Per-class precision/recall and the confusion matrix tell you *what kind* of mistakes the model is making.
- **Validator comments are the most valuable artifact in this whole notebook.** Numbers tell you *that* something is wrong; human notes tell you *what* and *why*.
- **Prompt iteration is a measure → diagnose → fix → re-measure loop.** Without the dataset and the metrics, prompt tweaks are just vibes.

### Suggested next steps
- Replace the synthetic seed tickets with anonymized **real** tickets from your inbox.
- Upload the dataset to LangSmith as a versioned **Dataset** and use the LangSmith `evaluate()` runner so each prompt version gets a saved score.
- Add a second node to the graph — e.g., a confidence check that routes low-confidence predictions to a human queue.